# Industrial Machine Condition Classification
## Multiclass Supervised Machine Learning Pipeline & Competition Benchmark

---

### Project Overview
The objective of this project is to develop an end-to-end, high-performance machine learning pipeline 
that classifies the operational condition of industrial machinery based on multivariate physical telemetry.

**3 Operational Target States (`condition`):**
* `0 — Normal`: Healthy machine operating within nominal tolerances (50.0% prevalence).
* `1 — Thermal Failure`: Failure driven by thermal stress, process overheating, and coolant deficit (25.0% prevalence).
* `2 — Mechanical Failure`: Failure driven by bearing wear, torque overload, and rotational vibration (25.0% prevalence).

**Colab Quickstart:**
1. Make sure `student_dataset.csv`, `test.csv`, and `sample_submission.csv` are uploaded to the file browser on the left.
2. Run all cells (`Runtime` -> `Run all`). All dependencies will be installed automatically.

In [ ]:
# Step 1: Install required packages (runs smoothly on Google Colab or local environment)
!pip install -q lightgbm xgboost catboost reportlab scikit-learn matplotlib seaborn

In [ ]:
# Step 2: Import Core Libraries & Global Configurations
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    cohen_kappa_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
)
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

SEED = 42
N_SPLITS = 5
CLASS_NAMES = ['0 - Normal', '1 - Thermal Failure', '2 - Mechanical Failure']
print(f'Pipeline Initialized: SEED = {SEED}, CV Folds = {N_SPLITS}')

In [ ]:
# Step 3: Auto-Detect and Load Datasets
def find_file(filename, candidate_dirs):
    for d in candidate_dirs:
        p = os.path.join(d, filename)
        if os.path.isfile(p):
            return p
    return None

search_dirs = ['.', './data', '../', './classification', './githubFinal', '/content']
train_path = find_file('train.csv', search_dirs) or find_file('student_dataset.csv', search_dirs)
test_path = find_file('test.csv', search_dirs)
sample_path = find_file('sample_submission.csv', search_dirs)

if train_path is None or test_path is None:
    raise FileNotFoundError(
        'train.csv (or student_dataset.csv) and test.csv not found! Please upload them to the current directory or files tab.'
    )

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_path) if sample_path else None

print(f'Train Data Shape: {train_df.shape} (from {train_path})')
print(f'Test Data Shape : {test_df.shape} (from {test_path})')
print('\nTarget Class Distribution (Condition):')
print(train_df['condition'].value_counts(normalize=True).mul(100).round(2).to_string())

In [ ]:
# Step 4: Exploratory Data Analysis Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
palette = ['#2ECC71', '#E74C3C', '#E67E22']

# 1. Target Class Counts
sns.countplot(data=train_df, x='condition', ax=axes[0], palette=palette)
axes[0].set_title('Target Class Distribution (2:1:1 Ratio)', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(['0: Normal (50%)', '1: Thermal (25%)', '2: Mechanical (25%)'])

# 2. Thermodynamic Coupling (Ambient vs Process Temp)
sample_plot = train_df.sample(min(3000, len(train_df)), random_state=42)
sns.scatterplot(
    data=sample_plot, x='ambient_temp', y='process_temp', hue='condition',
    palette=palette, alpha=0.6, ax=axes[1]
)
axes[1].set_title('Thermodynamic Heat Gradient (r ≈ 0.61)', fontsize=12, fontweight='bold')

# 3. Electric Motor Torque vs Speed
sns.scatterplot(
    data=sample_plot, x='shaft_speed', y='drive_torque', hue='condition',
    palette=palette, alpha=0.6, ax=axes[2]
)
axes[2].set_title('Motor Torque-Speed Power Curve', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Step 5: Modular, Leak-Free Preprocessor (CleanMachinePreprocessor)
class CleanMachinePreprocessor(BaseEstimator, TransformerMixin):
    NUMERICAL_COLS = [
        'ambient_temp', 'process_temp', 'coolant_flow', 'shaft_speed',
        'drive_torque', 'power_load', 'component_wear', 'thermal_stress',
        'mechanical_load', 'vibration_index',
    ]
    CATEGORICAL_COL = 'unit_type'
    TYPE_MAP = {'S': 0, 'M': 1, 'H': 2}
    EPSILON = 1e-4

    def __init__(self, include_one_hot=True, epsilon=1e-4):
        self.include_one_hot = include_one_hot
        self.epsilon = epsilon
        self.fitted_medians_ = {}
        self.fitted_mode_type_ = 'M'
        self.feature_names_ = []

    def fit(self, X, y=None):
        df = X.copy()
        for col in self.NUMERICAL_COLS:
            if col in df.columns:
                self.fitted_medians_[col] = float(df[col].median())
            else:
                self.fitted_medians_[col] = 0.0
        if self.CATEGORICAL_COL in df.columns:
            mode_vals = df[self.CATEGORICAL_COL].mode()
            self.fitted_mode_type_ = str(mode_vals.iloc[0]) if len(mode_vals) > 0 else 'M'
        else:
            self.fitted_mode_type_ = 'M'

        transformed_df = self._transform_internal(df)
        self.feature_names_ = [c for c in transformed_df.columns if c not in ['unit_id', 'condition', 'unit_type']]
        for col in self.feature_names_:
            if col not in self.fitted_medians_:
                self.fitted_medians_[col] = float(transformed_df[col].median())
        return self

    def _transform_internal(self, df):
        d = df.copy()
        # 1. Impute raw numerical features
        for col in self.NUMERICAL_COLS:
            if col in d.columns:
                d[col] = d[col].fillna(self.fitted_medians_.get(col, 0.0))
        # 2. Impute and encode categorical feature
        if self.CATEGORICAL_COL in d.columns:
            clean_type = d[self.CATEGORICAL_COL].fillna(self.fitted_mode_type_)
            d['unit_type_num'] = clean_type.map(self.TYPE_MAP).fillna(1).astype(int)
            if self.include_one_hot:
                d['unit_type_S'] = (clean_type == 'S').astype(int)
                d['unit_type_M'] = (clean_type == 'M').astype(int)
                d['unit_type_H'] = (clean_type == 'H').astype(int)
        else:
            d['unit_type_num'] = 1
            if self.include_one_hot:
                d['unit_type_S'], d['unit_type_M'], d['unit_type_H'] = 0, 1, 0

        # 3. Discovered Pairwise Non-Linear Sensor Boundaries
        d['shaft_speed_x_vibration_index'] = d['shaft_speed'] * d['vibration_index']
        d['drive_torque_x_component_wear'] = d['drive_torque'] * d['component_wear']
        d['thermal_stress_sub_vibration_index'] = d['thermal_stress'] - d['vibration_index']
        d['thermal_stress_div_vibration_index'] = d['thermal_stress'] / (d['vibration_index'] + self.epsilon)
        d['coolant_flow_div_power_load'] = d['coolant_flow'] / (d['power_load'].abs() + self.epsilon)
        d['process_temp_sub_coolant_flow'] = d['process_temp'] - d['coolant_flow']
        d['shaft_speed_x_mechanical_load'] = d['shaft_speed'] * d['mechanical_load']
        d['ambient_temp_x_process_temp'] = d['ambient_temp'] * d['process_temp']
        d['coolant_flow_x_mechanical_load'] = d['coolant_flow'] * d['mechanical_load']
        d['coolant_flow_x_shaft_speed'] = d['coolant_flow'] * d['shaft_speed']
        d['component_wear_div_thermal_stress'] = d['component_wear'] / (d['thermal_stress'].abs() + self.epsilon)
        d['drive_torque_div_thermal_stress'] = d['drive_torque'] / (d['thermal_stress'].abs() + self.epsilon)
        d['process_temp_sub_thermal_stress'] = d['process_temp'] - d['thermal_stress']
        d['coolant_flow_sub_drive_torque'] = d['coolant_flow'] - d['drive_torque']
        d['coolant_flow_sub_power_load'] = d['coolant_flow'] - d['power_load']

        # 4. Rotational Dynamics
        speed_norm = d['shaft_speed'] / 1000.0
        d['centrifugal_force'] = d['mechanical_load'] * (speed_norm ** 2) * d['vibration_index']
        d['rotational_kinetic_energy'] = 0.5 * d['mechanical_load'] * (speed_norm ** 2)

        # 5. Thermodynamics & Cooling Balance
        d['temp_gradient'] = d['process_temp'] - d['ambient_temp']
        d['power_dissipation_ratio'] = d['power_load'] / (d['coolant_flow'] + self.epsilon)

        # 6. Higher-Order Compound Physics Tensors
        d['friction_heat_flux'] = (d['drive_torque'] * d['shaft_speed'] * d['component_wear']) / ((d['coolant_flow'] + 1.0) * 1e4)
        d['shock_energy'] = (d['vibration_index'] ** 2) * d['shaft_speed'] * d['mechanical_load'] / 1e5

        # 7. Numerical Sanitization
        for col in d.columns:
            if col not in ['unit_id', 'condition', 'unit_type']:
                d[col] = d[col].replace([np.inf, -np.inf], np.nan)
                fallback = self.fitted_medians_.get(col, 0.0)
                d[col] = d[col].fillna(fallback)
        return d

    def transform(self, X):
        transformed_df = self._transform_internal(X)
        return transformed_df[self.feature_names_].values

print('CleanMachinePreprocessor defined successfully.')

In [ ]:
# Step 6: Identify Synthetic Boundary Noise & Configure Sample Weights
def compute_noise_weights(train_df):
    # Confident heuristic detecting simulation noise in synthetic dataset
    weights = np.ones(len(train_df), dtype=np.float32)
    wear = train_df['component_wear'].fillna(train_df['component_wear'].median())
    vib = train_df['vibration_index'].fillna(train_df['vibration_index'].median())
    stress = train_df['thermal_stress'].fillna(train_df['thermal_stress'].median())
    cond = train_df['condition'].values

    # Confident Normal but labeled Failure, or vice versa
    noise_mask = (
        ((wear < 50) & (vib < 1.0) & (stress < 50) & (cond != 0)) |
        ((wear > 250) & (vib > 4.5) & (cond == 0)) |
        ((stress > 300) & (cond == 0))
    )
    noise_count = int(noise_mask.sum())
    weights[noise_mask] = 0.25  # 75% down-weighting
    print(f'Identified {noise_count} contradictory simulation noise samples ({noise_count / len(train_df) * 100:.2f}%). Assigned weight = 0.25')
    return weights

sample_weights = compute_noise_weights(train_df)

In [ ]:
# Step 7: Metric Computation Helper
def compute_metrics(y_true, y_pred, y_prob=None):
    m = {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'cohen_kappa': float(cohen_kappa_score(y_true, y_pred)),
        'matthews_corrcoef': float(matthews_corrcoef(y_true, y_pred)),
    }
    if y_prob is not None:
        clipped = np.clip(y_prob, 1e-15, 1 - 1e-15)
        clipped = clipped / clipped.sum(axis=1, keepdims=True)
        m['log_loss'] = float(log_loss(y_true, clipped))
    else:
        m['log_loss'] = None
    return m

In [ ]:
# Step 8: 5-Fold Stratified Cross-Validation Training for All Models
X_raw = train_df.drop(columns=['unit_id', 'condition'])
y_train = train_df['condition'].values
X_test_raw = test_df.drop(columns=['unit_id']) if 'unit_id' in test_df.columns else test_df.copy()

model_names = [
    'LightGBM', 'XGBoost', 'CatBoost', 'RandomForest',
    'LogisticRegression', 'BayesClassifier', 'NaiveBayes', 'QuadraticBayes'
]

oof_probs = {m: np.zeros((len(train_df), 3), dtype=np.float64) for m in model_names}
test_probs = {m: np.zeros((len(test_df), 3), dtype=np.float64) for m in model_names}

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_raw, y_train), start=1):
    print(f'\n========== Training Fold {fold}/{N_SPLITS} ==========')
    X_tr_df, y_tr, w_tr = X_raw.iloc[tr_idx], y_train[tr_idx], sample_weights[tr_idx]
    X_val_df, y_val = X_raw.iloc[val_idx], y_train[val_idx]
    
    # Preprocessing (Fitted strictly on training fold)
    prep = CleanMachinePreprocessor(include_one_hot=True)
    prep.fit(X_tr_df)
    X_tr = prep.transform(X_tr_df)
    X_val = prep.transform(X_val_df)
    X_te = prep.transform(X_test_raw)
    
    # Scaled features for linear & generative models
    scaler = StandardScaler().fit(X_tr)
    X_tr_s = scaler.transform(X_tr)
    X_val_s = scaler.transform(X_val)
    X_te_s = scaler.transform(X_te)
    
    # 1. LightGBM
    clf_lgb = lgb.LGBMClassifier(
        n_estimators=1500, learning_rate=0.02, num_leaves=48, max_depth=7,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.15, reg_lambda=1.5,
        min_child_samples=30, random_state=SEED + fold, verbose=-1, n_jobs=-1
    )
    clf_lgb.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(60, verbose=False)])
    p_lgb = clf_lgb.predict_proba(X_val)
    oof_probs['LightGBM'][val_idx] = p_lgb
    test_probs['LightGBM'] += clf_lgb.predict_proba(X_te) / N_SPLITS
    
    # 2. XGBoost
    clf_xgb = XGBClassifier(
        n_estimators=1500, learning_rate=0.02, max_depth=6, subsample=0.8,
        colsample_bytree=0.8, min_child_weight=4, reg_alpha=0.15, reg_lambda=1.5,
        tree_method='hist', random_state=SEED + fold, n_jobs=-1, early_stopping_rounds=60
    )
    clf_xgb.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=[(X_val, y_val)], verbose=False)
    p_xgb = clf_xgb.predict_proba(X_val)
    oof_probs['XGBoost'][val_idx] = p_xgb
    test_probs['XGBoost'] += clf_xgb.predict_proba(X_te) / N_SPLITS
    
    # 3. CatBoost
    clf_cat = CatBoostClassifier(
        iterations=1500, learning_rate=0.025, depth=7, l2_leaf_reg=5.0,
        random_strength=0.8, random_seed=SEED + fold, verbose=0, early_stopping_rounds=60
    )
    clf_cat.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=(X_val, y_val), verbose=False)
    p_cat = clf_cat.predict_proba(X_val)
    oof_probs['CatBoost'][val_idx] = p_cat
    test_probs['CatBoost'] += clf_cat.predict_proba(X_te) / N_SPLITS
    
    # 4. Random Forest
    clf_rf = RandomForestClassifier(
        n_estimators=400, max_depth=19, min_samples_leaf=2, max_features=0.7,
        random_state=SEED + fold, n_jobs=-1
    )
    clf_rf.fit(X_tr, y_tr, sample_weight=w_tr)
    p_rf = clf_rf.predict_proba(X_val)
    oof_probs['RandomForest'][val_idx] = p_rf
    test_probs['RandomForest'] += clf_rf.predict_proba(X_te) / N_SPLITS
    
    # 5. Logistic Regression
    clf_lr = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', random_state=SEED + fold)
    clf_lr.fit(X_tr_s, y_tr, sample_weight=w_tr)
    p_lr = clf_lr.predict_proba(X_val_s)
    oof_probs['LogisticRegression'][val_idx] = p_lr
    test_probs['LogisticRegression'] += clf_lr.predict_proba(X_te_s) / N_SPLITS
    
    # 6. Bayes Classifier (LDA)
    clf_bayes = LinearDiscriminantAnalysis()
    clf_bayes.fit(X_tr_s, y_tr)
    p_bayes = clf_bayes.predict_proba(X_val_s)
    oof_probs['BayesClassifier'][val_idx] = p_bayes
    test_probs['BayesClassifier'] += clf_bayes.predict_proba(X_te_s) / N_SPLITS
    
    # 7. Naive Bayes (Gaussian NB)
    clf_nb = GaussianNB()
    clf_nb.fit(X_tr_s, y_tr, sample_weight=w_tr)
    p_nb = clf_nb.predict_proba(X_val_s)
    oof_probs['NaiveBayes'][val_idx] = p_nb
    test_probs['NaiveBayes'] += clf_nb.predict_proba(X_te_s) / N_SPLITS
    
    # 8. Quadratic Bayes (QDA)
    clf_qda = QuadraticDiscriminantAnalysis(reg_param=0.01)
    clf_qda.fit(X_tr_s, y_tr)
    p_qda = clf_qda.predict_proba(X_val_s)
    oof_probs['QuadraticBayes'][val_idx] = p_qda
    test_probs['QuadraticBayes'] += clf_qda.predict_proba(X_te_s) / N_SPLITS
    
    acc_lgb = accuracy_score(y_val, np.argmax(p_lgb, axis=1))
    acc_xgb = accuracy_score(y_val, np.argmax(p_xgb, axis=1))
    acc_cat = accuracy_score(y_val, np.argmax(p_cat, axis=1))
    acc_rf  = accuracy_score(y_val, np.argmax(p_rf, axis=1))
    print(f'Fold {fold} Acc -> LGB: {acc_lgb:.4f} | XGB: {acc_xgb:.4f} | CAT: {acc_cat:.4f} | RF: {acc_rf:.4f}')

print('\nAll 5 Folds Successfully Trained!')

In [ ]:
# Step 9: Multi-Model Ensembling, Probability Blending & Normal-Protection Calibration

# 1. Top-3 Equal Ensemble
top3_oof = (oof_probs['LightGBM'] + oof_probs['XGBoost'] + oof_probs['CatBoost']) / 3.0
top3_test = (test_probs['LightGBM'] + test_probs['XGBoost'] + test_probs['CatBoost']) / 3.0
oof_probs['Top3_Ensemble'] = top3_oof
test_probs['Top3_Ensemble'] = top3_test

# 2. Nelder-Mead Optimal Soft-Voting Blending Weights
def blend_objective(weights):
    w = np.clip(weights, 0, 1)
    if w.sum() == 0: return 1.0
    w = w / w.sum()
    blend = w[0] * oof_probs['LightGBM'] + w[1] * oof_probs['XGBoost'] + w[2] * oof_probs['CatBoost']
    return -accuracy_score(y_train, np.argmax(blend, axis=1))

res = minimize(blend_objective, [0.33, 0.33, 0.34], method='Nelder-Mead')
opt_w = np.clip(res.x, 0, 1)
opt_w = opt_w / opt_w.sum()
print(f'Optimal Soft-Voting Weights -> CatBoost: {opt_w[2]*100:.2f}%, XGBoost: {opt_w[1]*100:.2f}%, LightGBM: {opt_w[0]*100:.2f}%')

raw_oof_blend = opt_w[0] * oof_probs['LightGBM'] + opt_w[1] * oof_probs['XGBoost'] + opt_w[2] * oof_probs['CatBoost']
raw_test_blend = opt_w[0] * test_probs['LightGBM'] + opt_w[1] * test_probs['XGBoost'] + opt_w[2] * test_probs['CatBoost']

# 3. Normal-Protection Multipliers Calibration
M_normal, M_thermal, M_mechanical = 1.000, 0.9127, 1.0200
multipliers = np.array([M_normal, M_thermal, M_mechanical])

calib_oof = raw_oof_blend * multipliers
calib_oof = calib_oof / calib_oof.sum(axis=1, keepdims=True)
oof_probs['Calibrated_Ensemble'] = calib_oof

calib_test = raw_test_blend * multipliers
calib_test = calib_test / calib_test.sum(axis=1, keepdims=True)
test_probs['Calibrated_Ensemble'] = calib_test

best_acc = accuracy_score(y_train, np.argmax(calib_oof, axis=1))
print(f'\n★ Peak Calibrated Ensemble Out-Of-Fold Accuracy: {best_acc * 100:.4f}% ({best_acc:.5f}) ★')

In [ ]:
# Step 10: Model Evaluation Tables

all_evaluated = [
    'Calibrated_Ensemble', 'XGBoost', 'Top3_Ensemble', 'CatBoost',
    'LightGBM', 'RandomForest', 'LogisticRegression', 'BayesClassifier',
    'QuadraticBayes', 'NaiveBayes'
]

summary_rows = []
for m in all_evaluated:
    p_val = oof_probs[m]
    pred = np.argmax(p_val, axis=1)
    stats = compute_metrics(y_train, pred, p_val)
    stats['model'] = m
    summary_rows.append(stats)

summary_df = pd.DataFrame(summary_rows)

# Format required by syllabus: | Model | Best Score Achieved |
display_table_4 = summary_df[['model', 'accuracy']].copy()
display_table_4.columns = ['Model', 'Best Score Achieved']
display_table_4['Best Score Achieved'] = display_table_4['Best Score Achieved'].apply(lambda x: f'{x*100:.2f}% ({x:.5f})')

print('=' * 65)
print('REPORT SECTION 4: MODELS USED (BEST SCORE ACHIEVED)')
print('=' * 65)
display(display_table_4)

print('\n' + '=' * 65)
print('FULL MULTI-METRIC PERFORMANCE COMPARISON TABLE')
print('=' * 65)
full_display = summary_df[[
    'model', 'accuracy', 'balanced_accuracy', 'macro_f1',
    'weighted_f1', 'log_loss', 'cohen_kappa', 'matthews_corrcoef'
]].copy()
for col in ['accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1']:
    full_display[col] = full_display[col].apply(lambda x: f'{x*100:.2f}%')
for col in ['log_loss', 'cohen_kappa', 'matthews_corrcoef']:
    full_display[col] = full_display[col].apply(lambda x: f'{x:.4f}')
display(full_display)

In [ ]:
# Step 11: Confusion Matrix Diagnostics for Winning Calibrated Ensemble
best_oof_pred = np.argmax(oof_probs['Calibrated_Ensemble'], axis=1)
cm = confusion_matrix(y_train, best_oof_pred)

plt.figure(figsize=(7, 5.5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Pred Normal (0)', 'Pred Thermal (1)', 'Pred Mechanical (2)'],
    yticklabels=['True Normal (0)', 'True Thermal (1)', 'True Mechanical (2)']
)
plt.title('Calibrated Ensemble OOF Confusion Matrix (16,000 Samples)', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('Ground Truth Label')
plt.tight_layout()
plt.show()

print('\nClassification Report (Winning Calibrated Ensemble):')
print(classification_report(y_train, best_oof_pred, target_names=CLASS_NAMES, digits=4))

In [ ]:
# Step 12: Export Verified Kaggle Submissions
os.makedirs('predictions', exist_ok=True)

# 1. Winning Calibrated Ensemble Submission
final_test_pred = np.argmax(test_probs['Calibrated_Ensemble'], axis=1)
sub_df = pd.DataFrame({
    'unit_id': test_df['unit_id'],
    'condition': final_test_pred
})
sub_df.to_csv('submission.csv', index=False)
sub_df.to_csv('predictions/submission_Calibrated_Ensemble.csv', index=False)

# 2. Top-3 Equal Ensemble Submission
top3_pred = np.argmax(test_probs['Top3_Ensemble'], axis=1)
sub_top3 = pd.DataFrame({'unit_id': test_df['unit_id'], 'condition': top3_pred})
sub_top3.to_csv('top3ensemblesubmission.csv', index=False)
sub_top3.to_csv('predictions/submission_Top3_Ensemble.csv', index=False)

# 3. Individual Model Submissions
for m in model_names:
    m_pred = np.argmax(test_probs[m], axis=1)
    m_sub = pd.DataFrame({'unit_id': test_df['unit_id'], 'condition': m_pred})
    m_sub.to_csv(f'predictions/submission_{m}.csv', index=False)

print('=' * 65)
print('KAGGLE SUBMISSION INTEGRITY CHECK: submission.csv')
print('=' * 65)
print(f'File Size      : {os.path.getsize("submission.csv")} bytes')
print(f'Total Rows     : {len(sub_df)} (Target: 4,000)')
print(f'Null Values    : {sub_df.isnull().sum().sum()}')
print(f'Columns        : {list(sub_df.columns)}')
print('\nPredicted Class Distribution:')
print(sub_df['condition'].value_counts().rename(index={0: '0 - Normal', 1: '1 - Thermal', 2: '2 - Mechanical'}))
print('\nPercentages:')
print((sub_df['condition'].value_counts(normalize=True)*100).round(2).rename(index={0: '0 - Normal', 1: '1 - Thermal', 2: '2 - Mechanical'}))
print('\n✓ submission.csv is verified and ready for Kaggle submission!')

In [ ]:
# Step 13: Optional One-Click Download in Google Colab
try:
    from google.colab import files
    print('Google Colab detected. Downloading submission.csv...')
    files.download('submission.csv')
except Exception:
    print('Local execution: submission.csv saved in current working directory.')